# 02 · Data cleaning
**Where to Invest: Microsoft Growth Investment Analysis**

Goal: Clean every issue found in `01_profiling.ipynb` and write tidy tables to `data/clean/`, ready to load
into DuckDB. **Every rule and the rows it touches go in a cleaning log** (`data/clean/cleaning_log.csv`).

Steps:
1. Parse each system's dates and numbers
2. Convert every amount to USD
3. Restate credit memos to the month of the original invoice
4. Net off GL reversals and flag late postings
5. Restate the moved product line
6. Map CRM country names to ISO codes
7. Unpivot the budget

Tool: Python (pandas, numpy). Output columns use `snake_case`, dates use ISO format (`YYYY-MM-DD`).
The fiscal calendar is not a file: it is built in SQL (`sql/01_build_star_schema.sql`).

> **Synthetic data, calibrated to Microsoft's published results.** Generated by `data/generate.py`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(exist_ok=True)


def read_raw(name: str, **options) -> pd.DataFrame:
    """Read a raw export as text so nothing is converted silently."""
    return pd.read_csv(RAW / name, dtype=str, keep_default_na=False, **options)

## The cleaning log
Each call to `log()` records one rule: which file, what was wrong, what was done, and how many rows it touched.

In [2]:
cleaning_log = []


def log(file: str, issue: str, action: str, rows_affected: int, rows_after: int | None = None) -> None:
    cleaning_log.append({
        "step": len(cleaning_log) + 1,
        "file": file,
        "issue": issue,
        "action": action,
        "rows_affected": int(rows_affected),
        "rows_after": rows_after,
    })

## Parsing helpers
One small function per format, so each rule is easy to test and explain.

In [3]:
def parse_decimal_comma(s: pd.Series) -> pd.Series:
    """'1.234,56' -> 1234.56  (ERP billing, German locale)."""
    return s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)


def parse_trailing_minus(s: pd.Series) -> pd.Series:
    """'1,234.56-' -> -1234.56  (SAP list output)."""
    negative = s.str.endswith("-")
    value = s.str.rstrip("-").str.replace(",", "", regex=False).astype(float)
    return value.where(~negative, -value)


def parse_us_number(s: pd.Series) -> pd.Series:
    """'1,234.5' -> 1234.5; blanks become NaN."""
    return pd.to_numeric(s.str.replace(",", "", regex=False).replace("", np.nan))


def compact_to_iso(s: pd.Series) -> pd.Series:
    """'20250701' -> '2025-07-01'. Kept as text because 9999-12-31 is outside pandas' date range."""
    return s.str[:4] + "-" + s.str[4:6] + "-" + s.str[6:]


# quick self-tests
assert parse_decimal_comma(pd.Series(["1.234,56", "-12,50"])).tolist() == [1234.56, -12.5]
assert parse_trailing_minus(pd.Series(["1,234.56-", "10.00"])).tolist() == [-1234.56, 10.0]
assert parse_us_number(pd.Series(["1,234.5"])).tolist() == [1234.5]

## 1. Product hierarchy: restate the moved product line
Windows Commercial has two validity rows: More Personal Computing until 30 Jun 2025, then Productivity and
Business Processes. `segment` keeps the as-reported value; `segment_restated` gives every period today's
segment, for like-for-like comparisons.

In [4]:
ph = read_raw("product_hierarchy.csv")
products = pd.DataFrame({
    "material": ph["Material"],
    "material_description": ph["Material Description"],
    "product_line": ph["Product Line"],
    "profit_center": ph["Profit Center"],
    "segment": ph["Segment"],
    "revenue_type": ph["Revenue Type"],
    "base_unit": ph["Base Unit"],
    "valid_from": compact_to_iso(ph["Valid From"]),
    "valid_to": compact_to_iso(ph["Valid To"]),
})
current_segment = products[products["valid_to"] == "9999-12-31"].set_index("material")["segment"]
products["segment_restated"] = products["material"].map(current_segment)

moved = products[products["segment"] != products["segment_restated"]]
log("product_hierarchy", "Windows Commercial moved segment on 2025-07-01",
    "Added segment_restated (current segment for all periods)", len(moved), len(products))
moved

,material,material_description,product_line,profit_center,segment,revenue_type,base_unit,valid_from,valid_to,segment_restated
28,400380,Windows Enterprise E3,Windows Commercial,PC-WINC,More Personal Computing,Subscription,SEA,2000-01-01,2025-06-30,Productivity and Business Processes
30,400390,Windows 365 Cloud PC,Windows Commercial,PC-WINC,More Personal Computing,Subscription,SEA,2000-01-01,2025-06-30,Productivity and Business Processes


## 2. Regions

In [5]:
rg = read_raw("regions.csv")
regions = pd.DataFrame({
    "country_key": rg["Country Key"],
    "country_name": rg["Country Name"],
    "sales_area": rg["Sales Area"],
    "sales_region": rg["Sales Region"],
    "billing_currency": rg["Billing Currency"],
    "company_code": rg["Company Code"],
    "primary_cloud_region": rg["Primary Cloud Region"],
})
log("regions", "Column names with spaces", "Renamed to snake_case", 0, len(regions))

## 3. FX rates
Rates are units of foreign currency per 1 USD, so **USD amount = local amount ÷ rate**. USD has no row in
the treasury file, so a rate of 1.0 is added for every month.

In [6]:
fx_raw = read_raw("treasury_fx_rates.csv")
fx_rates = pd.DataFrame({
    "rate_type": fx_raw["Exchange Rate Type"],
    "currency": fx_raw["To Currency"],
    "month_start": pd.to_datetime(fx_raw["Valid From"], format="%Y%m%d").dt.to_period("M").dt.to_timestamp(),
    "rate_per_usd": fx_raw["Exchange Rate"].astype(float),
})
months = fx_rates["month_start"].drop_duplicates()
usd_rows = pd.DataFrame([{"rate_type": t, "currency": "USD", "month_start": m, "rate_per_usd": 1.0}
                         for t in ["M", "EOM"] for m in months])
fx_rates = pd.concat([fx_rates, usd_rows], ignore_index=True)
log("treasury_fx_rates", "USD has no rate row", "Added USD = 1.0 for every month and rate type", len(usd_rows), len(fx_rates))

# Monthly average rates are used for all conversions
fx_monthly = fx_rates[fx_rates["rate_type"] == "M"].set_index(["currency", "month_start"])["rate_per_usd"]
FX_FIRST, FX_LAST = months.min(), months.max()


def to_usd(local: pd.Series, currency: pd.Series, month: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Convert at the monthly average rate. Months outside the rate table use the nearest month."""
    month = month.clip(FX_FIRST, FX_LAST)
    rate = pd.Series(fx_monthly.reindex(pd.MultiIndex.from_arrays([currency, month])).values, index=local.index)
    return local / rate, rate

## 4. ERP billing: parse, restate segment, link credit memos, convert to USD

In [7]:
bl = read_raw("erp_billing_lines.csv", sep=";")
billing = pd.DataFrame({
    "billing_document": bl["Billing Document"],
    "item": bl["Item"],
    "billing_type": bl["Billing Type"],
    "billing_date": pd.to_datetime(bl["Billing Date"], format="%d.%m.%Y"),
    "customer_id": bl["Sold-To Party"],
    "sold_to_name": bl["Sold-To Name"].str.strip(),
    "country_key": bl["Country"],
    "material": bl["Material"],
    "region_code": bl["Plant"].replace("", np.nan),
    "quantity": bl["Billed Quantity"].astype(int),
    "sales_unit": bl["Sales Unit"],
    "net_value_local": parse_decimal_comma(bl["Net Value"]),
    "currency": bl["Document Currency"],
    "reference_document": bl["Reference Document"].replace("", np.nan),
})
billing["month_start"] = billing["billing_date"].dt.to_period("M").dt.to_timestamp()
billing["is_credit_memo"] = billing["billing_type"] == "G2"
RAW_BILLING_TOTAL = billing["net_value_local"].sum()             # used by the validation step
log("erp_billing_lines", "';' separator, DD.MM.YYYY dates, decimal comma", "Parsed to date and float", len(billing))
log("erp_billing_lines", "Sold-To Name has trailing spaces", "Trimmed", (bl["Sold-To Name"] != billing["sold_to_name"]).sum())

**Product and segment.** Each line takes the product-hierarchy row valid on its billing date, so
`segment_as_reported` follows the July 2025 move and `segment_restated` does not.

In [8]:
billing["billing_date_iso"] = billing["billing_date"].dt.strftime("%Y-%m-%d")
billing = billing.merge(products, on="material", how="left")
billing = billing[(billing["billing_date_iso"] >= billing["valid_from"]) &
                  (billing["billing_date_iso"] <= billing["valid_to"])]
billing = billing.rename(columns={"segment": "segment_as_reported"}).drop(
    columns=["billing_date_iso", "valid_from", "valid_to", "material_description", "base_unit"])
restated_lines = (billing["segment_as_reported"] != billing["segment_restated"]).sum()
log("erp_billing_lines", "Segment changed for Windows Commercial", "Added segment_as_reported and segment_restated",
    restated_lines, len(billing))

**Credit memos.** Each G2 line points to its original invoice. `revenue_month` is the month of the original
invoice (margin view); `month_start` stays the month the credit was booked (cash view).

In [9]:
invoice_dates = (billing[~billing["is_credit_memo"]]
                 .drop_duplicates("billing_document").set_index("billing_document")["billing_date"])
billing["original_billing_date"] = billing["reference_document"].map(invoice_dates)
billing["revenue_month"] = (billing["original_billing_date"].fillna(billing["billing_date"])
                            .dt.to_period("M").dt.to_timestamp())
log("erp_billing_lines", "Credit memos booked 1-3 months after the invoice",
    "Added original_billing_date and revenue_month (month of the original invoice)",
    (billing["revenue_month"] != billing["month_start"]).sum())

**USD.** Converted at the monthly average rate of the billing month.

In [10]:
billing["net_value_usd"], billing["fx_rate"] = to_usd(billing["net_value_local"], billing["currency"], billing["month_start"])
log("erp_billing_lines", "8 billing currencies", "Converted to USD at monthly average rate",
    (billing["currency"] != "USD").sum(), len(billing))
billing.head(3)

,billing_document,item,billing_type,billing_date,customer_id,sold_to_name,country_key,material,region_code,quantity,sales_unit,net_value_local,currency,reference_document,month_start,is_credit_memo,product_line,profit_center,segment_as_reported,revenue_type,segment_restated,original_billing_date,revenue_month,net_value_usd,fx_rate
0,90000000,000010,F2,2024-01-31,0020000000,METROWOOD MEDIA LTD,AE,400390,NaN,4026,SEA,108714.66,USD,NaN,2024-01-01,False,Windows Commercial,PC-WINC,More Personal Computing,Subscription,Productivity and Business Processes,NaT,2024-01-01,108714.660000,1.00000
2,90000001,000010,F2,2024-01-31,0020000026,KESTRELCREST BIOTECH PVT LTD,IN,400120,NaN,24907,SEA,19070639.47,INR,NaN,2024-01-01,False,Microsoft 365 Commercial,PC-M365C,Productivity and Business Processes,Subscription,Productivity and Business Processes,NaT,2024-01-01,224165.572167,85.07390
3,90000002,000010,F2,2024-01-31,0020000052,"Sunhaven Foods Co., Ltd.",JP,400140,NaN,5030,SEA,14102968.80,JPY,NaN,2024-01-01,False,Microsoft 365 Commercial,PC-M365C,Productivity and Business Processes,Subscription,Productivity and Business Processes,NaT,2024-01-01,95571.205754,147.56504


## 5. CRM opportunities
The CRM stores the country as a **full name** (`United States`); every other system uses ISO codes (`US`).
`Customer ID` is the ERP sold-to party, so won deals link to billing.

In [11]:
op = read_raw("crm_opportunities.csv")
opportunities = pd.DataFrame({
    "opportunity_id": op["Opportunity ID"],
    "topic": op["Topic"],
    "customer_id": op["Customer ID"],
    "customer_name": op["Potential Customer"],
    "customer_type": op["Customer Type"],
    "country_name": op["Country/Region"],
    "product_line": op["Product Line"],
    "opportunity_type": op["Opportunity Type"],
    "status": op["Status"],
    "status_reason": op["Status Reason"],
    "sales_stage": op["Sales Stage"],
    "probability": op["Probability (%)"].astype(float) / 100,
    "est_revenue_local": parse_us_number(op["Est. Revenue"]),
    "currency": op["Currency"],
    "created_on": pd.to_datetime(op["Created On"], format="%m/%d/%Y"),
    "est_close_date": pd.to_datetime(op["Est. Close Date"], format="%m/%d/%Y"),
    "actual_close_date": pd.to_datetime(op["Actual Close Date"].replace("", np.nan), format="%m/%d/%Y"),
    "owner": op["Owner"],
})
log("crm_opportunities", "US dates; revenue as '1,234.56' text", "Parsed to date and float", len(opportunities))

opportunities["country_key"] = opportunities["country_name"].map(regions.set_index("country_name")["country_key"])
log("crm_opportunities", "Country stored as a full name, not an ISO code", "Mapped to country_key through regions",
    opportunities["country_key"].notna().sum())

opportunities["close_date"] = opportunities["actual_close_date"].fillna(opportunities["est_close_date"])
close_month = opportunities["close_date"].dt.to_period("M").dt.to_timestamp()
opportunities["est_revenue_usd"], opportunities["fx_rate"] = to_usd(
    opportunities["est_revenue_local"], opportunities["currency"], close_month)
opportunities["weighted_revenue_usd"] = opportunities["est_revenue_usd"] * opportunities["probability"]
log("crm_opportunities", "Deals closing before Jan 2024 or after Sep 2026 have no FX rate",
    "Converted at the nearest available month", ((close_month < FX_FIRST) | (close_month > FX_LAST)).sum())

## 6. Customers
There is no customer master file. The customer list is built from the two systems that know customers:
billing (name and country of every billed customer) and the CRM (customer type, and prospects that never bought).

In [12]:
billed = (billing.groupby("customer_id")
          .agg(customer_name=("sold_to_name", "first"), country_key=("country_key", "first"))
          .reset_index())
crm = (opportunities.sort_values("created_on")
       .groupby("customer_id")
       .agg(crm_name=("customer_name", "last"), crm_country=("country_key", "last"),
            customer_type=("customer_type", "last"))
       .reset_index())
customers = billed.merge(crm, on="customer_id", how="outer")
customers["customer_name"] = customers["customer_name"].fillna(customers["crm_name"])
customers["country_key"] = customers["country_key"].fillna(customers["crm_country"])
customers["customer_type"] = customers["customer_type"].fillna("Not in CRM")
customers["is_billed"] = customers["customer_id"].isin(billed["customer_id"])
customers = customers[["customer_id", "customer_name", "country_key", "customer_type", "is_billed"]]
log("erp_billing_lines / crm_opportunities", "No customer master file",
    "Built customers from billed sold-to parties plus CRM prospects", len(customers), len(customers))
customers["customer_type"].value_counts()

customer_type
SMB              5330
Enterprise       2281
Not in CRM       2264
Public Sector    1789
Advertiser        981
Name: count, dtype: int64

## 7. General ledger: parse, period month, late postings, net off reversals
The **fiscal period** decides the month a cost belongs to (P01 = July), not the posting date.
Datacenter cost centres end with their cloud region (`AZR-DCO-US-EUS` → `EUS`).

In [13]:
gl_raw = read_raw("gl_cost_postings.csv")
gl = pd.DataFrame({
    "company_code": gl_raw["Company Code"],
    "document_number": gl_raw["Document Number"],
    "document_type": gl_raw["Document Type"],
    "posting_date": pd.to_datetime(gl_raw["Posting Date"], format="%d.%m.%Y"),
    "fiscal_year_num": gl_raw["Fiscal Year"].astype(int),
    "fiscal_period": gl_raw["Posting Period"].astype(int),
    "gl_account": gl_raw["G/L Account"],
    "gl_account_text": gl_raw["G/L Account Text"],
    "cost_center_id": gl_raw["Cost Center"].replace("", np.nan),
    "cost_center_name": gl_raw["Cost Center Name"].replace("", np.nan),
    "profit_center": gl_raw["Profit Center"],
    "functional_area": gl_raw["Functional Area"],
    "amount_local": parse_trailing_minus(gl_raw["Amount in LC"]),
    "currency": gl_raw["Local Currency"],
    "posting_text": gl_raw["Text"],
    "reversed_with": gl_raw["Reversed With"].replace("", np.nan),
})
RAW_GL_TOTAL = gl["amount_local"].sum()
log("gl_cost_postings", "DD.MM.YYYY dates; trailing minus '1,234.56-'", "Parsed to date and float", len(gl))

calendar_month = (gl["fiscal_period"] + 5) % 12 + 1
calendar_year = gl["fiscal_year_num"] - (gl["fiscal_period"] <= 6)
gl["month_start"] = pd.to_datetime(dict(year=calendar_year, month=calendar_month, day=1))
gl["is_late_posting"] = gl["posting_date"] > gl["month_start"] + pd.offsets.MonthEnd(0)
log("gl_cost_postings", "Late postings into an already-closed period",
    "Kept in their fiscal period; flagged is_late_posting", gl["is_late_posting"].sum())

gl["product_line"] = gl["profit_center"].map(
    products.drop_duplicates("profit_center").set_index("profit_center")["product_line"]).fillna("Corporate")
is_dc = gl["cost_center_name"].str.contains("-DCO-", na=False)
gl["region_code"] = np.where(is_dc, gl["cost_center_name"].str.split("-").str[-1], None)
log("gl_cost_postings", "Blank cost centre (id and name)", "Kept; product line taken from profit centre; no region",
    gl["cost_center_id"].isna().sum())

**Reversals.** Every reversed document and its reversal point to each other (`reversed_with`).
- Pairs **in the same period** are mistakes that were corrected (wrong amount, then reversed): they net
  to zero, so both lines are removed. The corrected re-posting stays.
- Pairs **across two periods** are month-end accruals reversed on day 1 of the next month. That is correct
  accrual accounting, so they are kept.

In [14]:
pairs = gl.dropna(subset=["reversed_with"]).copy()
pairs["pair_id"] = np.where(pairs["document_number"] < pairs["reversed_with"],
                            pairs["document_number"] + "-" + pairs["reversed_with"],
                            pairs["reversed_with"] + "-" + pairs["document_number"])
pair_stats = pairs.groupby("pair_id").agg(lines=("amount_local", "size"),
                                          net=("amount_local", "sum"),
                                          periods=("month_start", "nunique"))
same_period_pairs = pair_stats[(pair_stats["lines"] == 2) & (pair_stats["periods"] == 1)
                               & (pair_stats["net"].abs() < 0.05)].index
drop = pairs["pair_id"].isin(same_period_pairs)
removed_net = pairs.loc[drop, "amount_local"].sum()

gl = gl.drop(index=pairs.index[drop])
log("gl_cost_postings", "Wrong postings reversed in the same period",
    "Removed both lines of each pair (net zero)", drop.sum(), len(gl))
log("gl_cost_postings", "Accruals reversed in the next period",
    "Kept (correct accrual accounting)", (~drop).sum())

gl["amount_usd"], gl["fx_rate"] = to_usd(gl["amount_local"], gl["currency"], gl["month_start"])
log("gl_cost_postings", "Amounts in local currency", "Converted to USD at monthly average rate",
    (gl["currency"] != "USD").sum(), len(gl))
print(f"Reversal pairs removed: {len(same_period_pairs):,} (net local amount {removed_net:,.2f})")

Reversal pairs removed: 820 (net local amount 0.00)


## 8. Datacenter regions: capex and capacity
One row per cloud region × month. Capex is in USD; percentages arrive as 0–100.

In [15]:
dr = read_raw("datacenter_regions.csv")
datacenter = pd.DataFrame({
    "region_code": dr["Region Code"],
    "region_name": dr["Datacenter Region"],
    "country_key": dr["Country"],
    "month_start": pd.to_datetime(dr["Month"], format="%Y-%m"),
    "capex_build_usd": dr["Capex Datacenter Build (USD)"].astype(float),
    "capex_fleet_usd": dr["Capex Server and GPU Fleet (USD)"].astype(float),
    "capex_network_usd": dr["Capex Network (USD)"].astype(float),
    "capex_finance_lease_usd": dr["of which Finance Leases (USD)"].astype(float),
    "installed_mw": dr["Installed Capacity (MW)"].astype(float),
    "used_mw": dr["Used Capacity (MW)"].astype(float),
    "utilization": dr["Utilization %"].astype(float) / 100,
    "new_capacity_mw": dr["New Capacity Online (MW)"].astype(float),
    "availability": dr["Availability %"].astype(float) / 100,
})
datacenter["capex_usd"] = datacenter[["capex_build_usd", "capex_fleet_usd", "capex_network_usd"]].sum(axis=1)
datacenter["capex_owned_usd"] = datacenter["capex_usd"] - datacenter["capex_finance_lease_usd"]
log("datacenter_regions", "YYYY-MM months; percentages as 0-100", "Parsed months; converted percentages to 0-1",
    len(datacenter), len(datacenter))

## 9. Budget: drop subtotals and unpivot
The planning export has 2 metadata lines, one column per month, segment subtotal rows and values in USD
thousands. The clean table has one row per version × product line × line item × month, in USD.

In [16]:
bd = read_raw("fpa_budget.csv", skiprows=2)
subtotals = bd["Product Line"].str.startswith("Total ")
bd = bd[~subtotals]
log("fpa_budget", "Segment subtotal rows would double count", "Removed 'Total ...' rows", subtotals.sum(), len(bd))

MONTH_COLUMNS = ["Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]
budget = bd.melt(id_vars=["Version", "Fiscal Year", "Segment", "Product Line", "Line Item"],
                 value_vars=MONTH_COLUMNS, var_name="month_name", value_name="amount_text")
month_number = pd.to_datetime(budget["month_name"], format="%b").dt.month
fiscal_year_num = 2000 + budget["Fiscal Year"].str[-2:].astype(int)
budget["month_start"] = pd.to_datetime(dict(year=fiscal_year_num - (month_number >= 7), month=month_number, day=1))
budget["amount_usd"] = parse_us_number(budget["amount_text"]) * 1000
budget = budget.rename(columns={"Version": "version", "Fiscal Year": "fiscal_year", "Segment": "segment",
                                "Product Line": "product_line", "Line Item": "line_item"})
budget = budget[["version", "fiscal_year", "segment", "product_line", "line_item", "month_start", "amount_usd"]]
log("fpa_budget", "Wide format (one column per month), USD thousands",
    "Unpivoted to one row per month; multiplied by 1,000", len(budget), len(budget))
budget.head(3).style.format({
    "amount_usd": lambda value: f"${value / 1_000_000_000:,.2f}B"
})

,version,fiscal_year,segment,product_line,line_item,month_start,amount_usd
0,Budget,FY25,Intelligent Cloud,Azure,Revenue,2024-07-01 00:00:00,$5.13B
1,Budget,FY25,Intelligent Cloud,Azure,Cost of Revenue,2024-07-01 00:00:00,$2.02B
2,Budget,FY25,Intelligent Cloud,Azure,Research and Development,2024-07-01 00:00:00,$0.60B


## 10. Validation
Cleaning must not change the totals it is not meant to change.

In [17]:
checks = pd.DataFrame([
    ("Billing local total unchanged", np.isclose(billing["net_value_local"].sum(), RAW_BILLING_TOTAL)),
    ("Every billing line has a product", billing["product_line"].notna().all()),
    ("Every billing line has an FX rate", billing["fx_rate"].notna().all()),
    ("No billing line duplicated by the joins", not billing.duplicated(["billing_document", "item"]).any()),
    ("Every credit memo found its original invoice", billing.loc[billing["is_credit_memo"], "original_billing_date"].notna().all()),
    ("GL total changed only by removed pairs (~0)", np.isclose(gl["amount_local"].sum() + removed_net, RAW_GL_TOTAL)),
    ("Every GL line has an FX rate", gl["fx_rate"].notna().all()),
    ("Every GL datacenter region exists in datacenter_regions", set(gl["region_code"].dropna()) <= set(datacenter["region_code"])),
    ("Every opportunity has a country_key", opportunities["country_key"].notna().all()),
    ("Every opportunity has an FX rate", opportunities["fx_rate"].notna().all()),
    ("Datacenter: one row per region and month", not datacenter.duplicated(["region_code", "month_start"]).any()),
    ("Budget rows = 240 product-line rows x 12 months", len(budget) == 240 * 12),
], columns=["check", "passed"])
assert checks["passed"].all(), checks[~checks["passed"]]
checks

,check,passed
0,Billing local total unchanged,True
1,Every billing line has a product,True
2,Every billing line has an FX rate,True
3,No billing line duplicated by the joins,True
4,Every credit memo found its original invoice,True
5,GL total changed only by removed pairs (~0),True
6,Every GL line has an FX rate,True
7,Every GL datacenter region exists in datacente...,True
8,Every opportunity has a country_key,True
9,Every opportunity has an FX rate,True


## 11. Save the clean tables and the cleaning log

In [18]:
outputs = {
    "products.csv": products,
    "regions.csv": regions,
    "fx_rates.csv": fx_rates,
    "customers.csv": customers,
    "opportunities.csv": opportunities,
    "billing.csv": billing,
    "gl_costs.csv": gl,
    "datacenter.csv": datacenter,
    "budget.csv": budget,
}
for name, df in outputs.items():
    df.to_csv(CLEAN / name, index=False, date_format="%Y-%m-%d")

cleaning_log_df = pd.DataFrame(cleaning_log).astype({"rows_after": "Int64"})
cleaning_log_df.to_csv(CLEAN / "cleaning_log.csv", index=False)

pd.DataFrame({"file": list(outputs), "rows": [len(df) for df in outputs.values()]})

,file,rows
0,products.csv,42
1,regions.csv,39
2,fx_rates.csv,528
3,customers.csv,12645
4,opportunities.csv,41612
5,billing.csv,306684
6,gl_costs.csv,118481
7,datacenter.csv,1122
8,budget.csv,2880


In [19]:
cleaning_log_df

,step,file,issue,action,rows_affected,rows_after
0,1,product_hierarchy,Windows Commercial moved segment on 2025-07-01,Added segment_restated (current segment for al...,2,42
1,2,regions,Column names with spaces,Renamed to snake_case,0,39
2,3,treasury_fx_rates,USD has no rate row,Added USD = 1.0 for every month and rate type,66,528
3,4,erp_billing_lines,"';' separator, DD.MM.YYYY dates, decimal comma",Parsed to date and float,306684,<NA>
4,5,erp_billing_lines,Sold-To Name has trailing spaces,Trimmed,45509,<NA>
5,6,erp_billing_lines,Segment changed for Windows Commercial,Added segment_as_reported and segment_restated,10417,306684
6,7,erp_billing_lines,Credit memos booked 1-3 months after the invoice,Added original_billing_date and revenue_month ...,2315,<NA>
7,8,erp_billing_lines,8 billing currencies,Converted to USD at monthly average rate,164814,306684
8,9,crm_opportunities,"US dates; revenue as '1,234.56' text",Parsed to date and float,41612,<NA>
9,10,crm_opportunities,"Country stored as a full name, not an ISO code",Mapped to country_key through regions,41612,<NA>


## Summary
- Every date and number is parsed from its system's own format into proper types.
- Every amount has a USD value at the monthly average rate.
- Credit memos carry the month of the invoice they correct.
- GL corrections that reverse each other in the same period are netted off; accruals are kept; late postings are flagged.
- Windows Commercial has both an as-reported and a restated segment.
- CRM countries are mapped to ISO codes; a customer list is built from billing and the CRM.
- The budget is in long format, in USD, without subtotals.

**Next:** `sql/01_build_star_schema.sql` loads `data/clean/` into DuckDB and builds the star schema.